# Enroll a new device from a wet/dry capture pair

TONE3000's model capture asks for exactly one recording: your device's response
("**wet**") to the standard NAM capture signal ("**dry**", e.g. `v3_0_0.wav`).
This notebook fits an **embedding** for that device against a frozen
one-to-many run — the network never changes, only a new conditioning vector is
optimized on the pair (Phase 5, `openamp.emulate.enroll.enroll_pair`).

1. Point `DRY_PATH` / `WET_PATH` at your pair and pick the run to enroll against.
   **Record the whole NAM signal, leading blips included** — we align on the
   blip, exactly like NAM's trainer does.
2. Run all cells: the pair is aligned (reamp latency, via the blip), level-matched
   to the corpus, split by time into train/val regions, and enrolled.
3. Listen: dry → wet (target) → model prediction on the held-back val region.

- Kernel: the **open-amp3000** conda env (Python 3.10).
- Runs on **CPU** by default so it never competes with a live training run.
- Artifacts land in `<run>/enroll/pairs/<NAME>/` (`enrolled_pair.pt` + metrics);
  rebuild the playable model any time with `load_pair_model(RUN_DIR, NAME)`.


In [ ]:
import os
from pathlib import Path

# run from the repo root so the config's relative data paths resolve
if not Path('configs').is_dir() and (Path.cwd().parent / 'configs').is_dir():
    os.chdir(Path.cwd().parent)
print('cwd:', Path.cwd())

import numpy as np

from openamp.core.config import load_config

# ---- inputs ---------------------------------------------------------------
DRY_PATH = Path('data/captures/v3_0_0.wav')      # the NAM capture signal you played
WET_PATH = Path('data/captures/my_amp_wet.wav')  # your device's recorded response
RUN_DIR  = Path('results/emulate/wavenet_a2')    # frozen run to enroll against
NAME     = 'my_amp'                              # artifact name (enroll/pairs/<NAME>/)

# ---- knobs ------------------------------------------------------------------
DEVICE        = 'cpu'    # 'cuda' when the GPU is free
PAIRS         = 500      # optimization windows per epoch
EPOCHS        = 30       # max; early-stopped on val ESR
LR            = 1e-2
VAL_FRAC      = 0.1      # last 10% of the pair is held back for val
NORMALIZE_DRY = True     # level-match the dry to the corpus (-18 dBFS RMS)

# reamp-latency alignment (see the notes cell):
#   'blip'   align on the capture's leading NAM blip — first-arrival, so it
#            strips interface latency but keeps the amp's tone. Needs the blip
#            in the take: play the WHOLE NAM signal, blips included. (default)
#   'xcorr'  whitened cross-correlation over the take — fallback with no blip;
#            carries the amp's group delay (~tens of samples).
#   'manual' use LAG exactly (best: your interface's reported round-trip).
ALIGN = 'blip'
LAG   = 0                # samples wet trails dry; only used when ALIGN='manual'

cfg = load_config()
print(f'run={RUN_DIR.name}  sample_rate={cfg.sample_rate}')


In [ ]:
import matplotlib.pyplot as plt

from openamp.core import constants as C
from openamp.dsp.audio import peak_dbfs, read_audio, resample, rms_dbfs
from openamp.emulate.enroll import blip_lag, estimate_lag

dry, sr = read_audio(DRY_PATH)
dry = resample(dry, sr, cfg.sample_rate)
wet, sr = read_audio(WET_PATH)
wet = resample(wet, sr, cfg.sample_rate)

# reamp chains add a fixed latency: correct it before enrolling (alignment is the
# #1 thing to get right — see the notes cell). VERIFY on the onset overlay below.
if ALIGN == 'manual':
    lag = int(LAG)
elif ALIGN == 'blip':
    lag = blip_lag(dry, wet, sample_rate=cfg.sample_rate)
else:
    lag = estimate_lag(dry, wet)
print(f"align={ALIGN}")
if lag > 0:
    wet = wet[lag:]
elif lag < 0:
    dry = dry[-lag:]
n = min(len(dry), len(wet))
dry, wet = dry[:n].copy(), wet[:n].copy()

# level-match the DRY to what the frozen net saw in training (corpus RMS target);
# the same gain goes on the wet so the pair stays one consistent input->output shot
if NORMALIZE_DRY:
    gain_db = C.TARGET_RMS_DBFS - rms_dbfs(dry)
    g = float(10 ** (gain_db / 20))
    dry *= g
    wet *= g
    print(f'applied {gain_db:+.1f} dB to both (dry -> {C.TARGET_RMS_DBFS} dBFS RMS)')

print(f'lag={lag:+d} samples ({1000 * lag / cfg.sample_rate:+.2f} ms)   '
      f'pair={n / cfg.sample_rate:.1f}s')
print(f'dry: rms {rms_dbfs(dry):6.1f} dBFS  peak {peak_dbfs(dry):6.1f} dBFS')
print(f'wet: rms {rms_dbfs(wet):6.1f} dBFS  peak {peak_dbfs(wet):6.1f} dBFS')

# left: 2 s overview.  right: sample-zoomed overlay at the sharpest transient,
# each trace peak-normalized so you can see whether the edges line up. A visible
# horizontal offset here = residual latency: adjust LAG by that many samples.
o = n // 3
end = min(o + 2 * cfg.sample_rate, n)
k = int(np.argmax(np.abs(np.diff(dry)))); w = 60      # sharpest dry edge, +-60 smp
z0, z1 = max(k - w, 0), min(k + w, n)
fig, (axl, axr) = plt.subplots(1, 2, figsize=(12, 2.8),
                               gridspec_kw={'width_ratios': [2, 1]})
axl.plot(np.arange(end - o) / cfg.sample_rate, dry[o:end], lw=0.6, alpha=0.8, label='dry')
axl.plot(np.arange(end - o) / cfg.sample_rate, wet[o:end], lw=0.6, alpha=0.8, label='wet')
axl.set_xlabel('s'); axl.set_title('aligned pair (2 s excerpt)'); axl.legend(loc='upper right')
nd = dry[z0:z1] / (np.max(np.abs(dry[z0:z1])) + 1e-9)
nw = wet[z0:z1] / (np.max(np.abs(wet[z0:z1])) + 1e-9)
axr.plot(np.arange(z0, z1) - k, nd, marker='.', ms=3, lw=0.8, label='dry')
axr.plot(np.arange(z0, z1) - k, nw, marker='.', ms=3, lw=0.8, label='wet')
axr.axvline(0, color='k', lw=0.5, alpha=0.4)
axr.set_xlabel('samples from transient'); axr.set_title('onset overlay (verify alignment)')
axr.legend(loc='upper right')
plt.tight_layout(); plt.show()


In [ ]:
from openamp.emulate.enroll import enroll_pair

metrics = enroll_pair(cfg, RUN_DIR, dry, wet, name=NAME, pairs=PAIRS,
                      epochs=EPOCHS, lr=LR, device=DEVICE, val_frac=VAL_FRAC,
                      sources={'dry': str(DRY_PATH), 'wet': str(WET_PATH),
                               'lag_samples': int(lag)})
metrics


In [ ]:
import pandas as pd
from IPython.display import Audio, display

from openamp.emulate.enroll import load_pair_model, render_dry

log = pd.read_csv(RUN_DIR / 'enroll' / 'pairs' / NAME / 'enroll_log.csv')
ax = log.plot(x='epoch', y='val_esr', marker='o', figsize=(7, 2.5), legend=False)
ax.set_ylabel('val ESR (pre-emph, pooled)')
ax.set_title('enrollment curve  (epoch -1 = table-mean baseline)')

model, blob = load_pair_model(RUN_DIR, NAME, device=DEVICE)
R = model.receptive_field
lo = n - int(round(metrics['val_seconds'] * cfg.sample_rate))
ctx = min(R, lo)                                  # real left-context from train side
pred = render_dry(model, dry[lo - ctx:], device=DEVICE)[ctx:]

print(f"val region ({metrics['val_seconds']:.1f}s):  "
      f"raw ESR {metrics['val_esr_render_raw']:.4f}   "
      f"pre-emph {metrics['val_esr_render_preemph']:.4f}")
for label, sig in (('dry (input)', dry[lo:]),
                   ('wet (target)', wet[lo:]),
                   ('prediction (enrolled)', pred)):
    print(label)
    display(Audio(sig, rate=cfg.sample_rate, normalize=False))


**Notes**

- **Baseline**: epoch −1 in the curve is the *table-mean* embedding — the
  frozen net's "generic amp". Enrollment must beat it; if it never does, the
  saved vector falls back to that prior (never worse than generic).
- **What good looks like**: holdout devices enrolled from full corpus renders
  reach the trained-device ESR ballpark. A single ~3 min sweep pair carries
  less audio diversity, so expect somewhat higher val ESR — the A/B render
  above is the honest check.
- **Alignment is the #1 thing to get right.** Embedding-only enrollment can't
  time-shift, so a constant wet/dry offset is error it can't absorb: in testing,
  a 37-sample (~0.8 ms) misalignment roughly *tripled* the val-render ESR, and a
  100-sample offset wrecked it.
- **Why `ALIGN='blip'` is the default.** The interface round-trip is a pure
  delay `L`; the amp's response to the blip *starts* immediately but its energy
  sits a few samples in (group delay). Aligning the blip's **first-arrival**
  recovers `L` alone and leaves the amp's group delay in the target for the
  model to learn. Whole-signal `estimate_lag` instead recovers `L + group_delay`
  and can't separate them (a ~10–40-sample residual). Blip alignment needs the
  blip in the take — play the entire NAM signal. Use `'xcorr'` only for
  blip-free material, and `'manual'` when your interface reports its latency.
- Whichever method: eyeball the onset overlay (the blip should sit on the black
  line) and trust the **val-render ESR** printed above as the final arbiter.
- The dry must be the *same* file you actually played into the device;
  alignment cannot fix clock drift or dropouts in the recording.
- `PAIRS` is an *optimization* budget (fresh random windows every epoch), not
  an audio budget: the same pair is re-windowed each epoch.
- The A/B players use `normalize=False` so loudness differences are real —
  keep capture peaks under 0 dBFS.
- The fitted vector is `[1, embedding_dim]` in
  `<run>/enroll/pairs/<NAME>/enrolled_pair.pt`, concat-compatible with the
  run's `embedding.pt` (future `emulate-demo --enrolled` hook).
